In [ ]:
import pandas as pd
import numpy as np
import sklearn
import matplotlib
import seaborn as sns
print("all good")

# Predicting breast cancer molecular subtype from gene expression

**Question:** Can a tumour's PAM50 molecular subtype (Luminal A, Luminal B, 
HER2-enriched, Basal-like) be predicted from its RNA-seq gene expression profile?

**Data:** TCGA-BRCA (expression matrix + clinical subtype labels)

**Approach:** supervised classification — start with a simple baseline model.

In [ ]:
import pandas as pd

expr = pd.read_csv('/Users/srij/Documents/TCGA BRCA project/Data/TCGA-BRCA.star_tpm.tsv.gz', sep='\t', index_col=0)
labels = pd.read_csv('/Users/srij/Documents/TCGA BRCA project/Data/brca_pam50', sep='\t', index_col=0)
# genes as rows and samples as columns

print("Expression shape: ", expr.shape)
print("Labels shape: ", labels.shape)
print(expr.iloc[:5, :5])
print(labels.head())



In [ ]:
print (set(expr.columns))
print (set(labels.index))

overlap = set(expr.columns) & set(labels.index)
print (len(overlap))

In [ ]:
print(list(labels.index)[:5])
print(list(expr.columns)[:5])
print("duplicate labels:", labels.index.duplicated().sum())

labels[~labels.index.duplicated(keep='first')] #getting rid of the duplicates but keeping the first occurence; list[~a] removes the a components 
print("label shape: ", labels.shape)

overlap = set(expr.columns) & set(labels.index) #set also drops duplicates - we make into a list so we can do the "&"" which lists cannot do
print("overlap", len(overlap))
overlap

In [ ]:
labels = labels[~labels.index.duplicated(keep='first')]
print("labels shape:", labels.shape)
overlap = set(expr.columns) & set(labels.index) #.index gives row labels and .columns gives column labels
print("overlap:", len(overlap))

In [ ]:
samples = list(overlap)
x = expr[samples].T
y = labels.loc[samples, 'PAM50'] #pulls out list of samples of the PAM50 column - returns just a list of either luma,b,basal,her
print(x.shape); print(y.shape); print(y.value_counts())

In [ ]:
gene_var = x.var() #picking most variation to prevent bias and overfitting, so we assign variation toa. variable and later order it
top_genes = gene_var.sort_values(ascending=False).head(1000).index #.head... --> takes top 1000 rows/genes and index pulls out the gene identifiers (ensembl IDs) - getting most variable
x_top=x[top_genes]
print(x_top.shape)

In [ ]:
from sklearn.model_selection import train_test_split
keep = ['LumA', 'LumB', 'Basal', 'Her2']
mask = y.isin(keep).values          # plain boolean array, no index to misalign - without the .values we get true and false but with respect to the index (sample), basically a table, the values make it a 1d list

x_top = x_top[mask] #list[True,False ,True] returns the 1st and 3rd elements
y = y[mask]
print(y.value_counts())
print(x_top.shape)
x_train, x_test, y_train, y_test = train_test_split (x_top, y, test_size=0.2, stratify=y, random_state=42)
#random state ensures that the training set remains to be the training set and the test stays the test and no mixing 
#stratify is to ensure proportion of each pam50 subtype matches with actual data - lumA dominates and Basal and Her are rarer and the split should reflect that
print(x_train.shape, x_test.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

model = LogisticRegression(max_iter=1000) #learns weights for each gene -> push a sample toward one subtype or another -> computes score -> passes through function that 
#turns into class probabilities --> highest prob is the prediction
# creates model with starting weights

model.fit(x_train, y_train) #training step - hands the model our training samples gene expression
# .fit() runs actual optimisation, repeatedly adjusting the weights up to 1000 iterations to reduce prediction error on training data

y_pred = model.predict(x_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
scores = cross_val_score(model, x_top, y, cv=5)
# splits into 5 groups --> trains on 4 and tests on 1 --> rotates so each fold gets its turn as the test set --> returns 5 accuracy score
print(scores)
print("mean:", scores.mean(), "std:", scores.std())
# get mean and standard deviation of the scores to make it more accurate

In [ ]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000)
model.fit(x_train, y_train)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

y_pred = model.predict(x_test)
cm = confusion_matrix(y_test, y_pred, labels=model.classes_)
ConfusionMatrixDisplay(cm, display_labels=model.classes_).plot()
plt.show()
# .classes_ returns array of subtype labels the model learned during .fit() - ['basal','her2', 'luma', 'lumb']
